In [ ]:
from json import dumps
from random import randrange, choice
dblen = {'user':20, 'group':10, 'item':30, 'unit':20, 'craft':20, 'robot':20}
dbup = {'user':['group'], 'item':['user']}
dbdown = {'user':['item'], 'group':['user']}
def dump(a):
    return dumps(a, separators=',:', sort_keys=True)
def db(obj, id):
    if id >= dblen[obj]:
        raise KeyError
    return {'id':id, 'name':f'{obj}{id}',
            **{i:dblen[i]*id//dblen[obj]for i in dbup.get(obj,())},
            **{i:[i for i in range(dblen[i]*id//dblen[obj],dblen[i]*(id+1)//dblen[obj])]for i in dbdown.get(obj,())}}
def req(method, obj, id, body):
    a = f'HTTP/1.0 {method} /{obj}'
    if method != 'POST':
        a += f'/{id}'
    if method in ('POST','PUT','PATCH'):
        a += '\n\n' + dump(body)
    return a
def rsp(method, obj, id, body):
    try:
        a = 'HTTP/1.0 200'
        if method == 'POST':
            body['id'] = dblen[obj]
        else:
            o = db(obj, id)
            if method == 'DELETE':
                return a
            if method == 'PUT':
                body['id'] = id
            else:
                if method == 'PATCH':
                    o.update(body)
                body = o
        a += '\n\n' + dump(body)
        return a
    except KeyError:
        return 'HTTP/1.0 404'
def dat(a=1, method=('GET','POST','PUT','DELETE','PATCH'), obj=('user','group','item','apple','unit','craft','robot'), id=range(25), body=10):
    t = []
    T = []
    for a in range(a):
        a = choice(method), choice(obj), choice(id), {'name':f'apple{randrange(body)}'}
        t.append(req(*a))
        T.append(rsp(*a))
    return t, T

In [ ]:
import torch
from torch.nn import *
class Lang:
    def __init__(s, a):
        s.a = '<bos>','<eos>',*a
        s.A = {a:i for i,a in enumerate(s.a)}
    def __len__(s):
        return len(s.a)
    def __call__(s, a):
        return torch.tensor([[0 if i<0else s.A[a[i]]if i<len(a)else 1for a in a]for i in range(-1,max(map(len,a))+1)], device='cuda')
    def __getitem__(s, a):
        for a in a.T.tolist():
            t = []
            for a in a:
                if a==1:
                    break
                t.append(s.a[a])
            yield ''.join(t)
class Seq2Seq(Module):
    def __init__(s, a, k, K):
        super().__init__()
        s.a = Lang(a)
        s.emb = Embedding(len(s.a),k)
        s.enc = GRU(k,K)
        s.mid = Sequential(Linear(K,K), ReLU(1), Linear(K,K))
        s.dec = GRU(k,K)
        s.deb = Sequential(Linear(K,k), ReLU(1), Linear(k,len(s.a)))
        s.cuda()
    def forward(s, x, y=65, k=.5):
        x = s.a(x)
        x = s.emb(x)
        if isinstance(y, int):
            y = torch.zeros(y+1,*x.shape[1:], device='cuda')
        else:
            if isinstance(y, list):
                y = s.a(y)[:-1]
            y = s.emb(y)
            y[torch.rand(len(y))<k] = 0
        x = s.enc(x)[1]
        x = s.mid(x)
        x = s.dec(y, x)[0]
        x = s.deb(x)
        return x
    def score(s, x, y, **_):
        y = s.a(y)
        x = s(x, y[:-1], **_)
        return functional.cross_entropy(x.transpose(1,2), y[1:])
    def predict(s, x, y=65, **_):
        return list(s.a[s(x,y,**_).argmax(2)])

In [ ]:
from tabulate import tabulate
from IPython.display import HTML
def box(a):
    display(HTML(tabulate(a,'keys','html').replace('>\n','>').replace('\n','<br>')))

In [ ]:
m = Seq2Seq(set(i for i in dat(999)for i in i for i in i), 8, 128)
o = torch.optim.Adam(m.parameters())
O = torch.optim.lr_scheduler.OneCycleLR(o, .002, 200000)
for i in range(10):
    for i in range(10):
        for i in range(O.total_steps//100):
            i = m.score(*dat(32))
            i.backward()
            o.step()
            o.zero_grad()
            O.step()
        print(end=f'{i:.3f} ')
    i,j = dat(4)
    with torch.no_grad():
        box({'x':i,'t':j,'y k=0':m.predict(i,j,k=0),'y':m.predict(i,j),'y k=1':m.predict(i,j,k=1)})